### Imports

In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Dataset, Subset
from PIL import Image
import numpy as np
import random
from sklearn.model_selection import train_test_split

### Data

In [2]:
train_dir = 'train_v1/' 
class CustomDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []
        for label, sub_dir in enumerate(os.listdir(root_dir)):
            print(label)
            full_dir = os.path.join(root_dir, sub_dir)
            for img_name in os.listdir(full_dir):
                img_path = os.path.join(full_dir, img_name)
                self.image_paths.append(img_path)
                self.labels.append(label)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image)
        return image, label

# Data preprocessing
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

dataset = CustomDataset(train_dir, transform=transform)
indicies = list(range(len(dataset)))
train_idx, test_idx = train_test_split(indicies, test_size=0.2, shuffle=True)

train_dataset = Subset(dataset, train_idx)
test_dataset = Subset(dataset, test_idx)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)

0
1


### Model

In [3]:
from torchvision.models import efficientnet_b0
from torchinfo import summary
efficientNet = efficientnet_b0()
summary(efficientNet)

Layer (type:depth-idx)                                  Param #
EfficientNet                                            --
├─Sequential: 1-1                                       --
│    └─Conv2dNormActivation: 2-1                        --
│    │    └─Conv2d: 3-1                                 864
│    │    └─BatchNorm2d: 3-2                            64
│    │    └─SiLU: 3-3                                   --
│    └─Sequential: 2-2                                  --
│    │    └─MBConv: 3-4                                 1,448
│    └─Sequential: 2-3                                  --
│    │    └─MBConv: 3-5                                 6,004
│    │    └─MBConv: 3-6                                 10,710
│    └─Sequential: 2-4                                  --
│    │    └─MBConv: 3-7                                 15,350
│    │    └─MBConv: 3-8                                 31,290
│    └─Sequential: 2-5                                  --
│    │    └─MBConv: 3-9         

In [ ]:
#  Training function
def train_model(model, train_loader, test_loader, criterion, optimizer, device, num_epochs=10):
    max_accuracy = 0 #  Print the highest accuracy.   
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device).float().view(-1, 1)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * inputs.size(0)
        epoch_loss = running_loss / len(train_loader.dataset)
        train_accuracy = eval_model(model,test_loader, device)    
        log_message = f'Epoch {epoch+1}/{num_epochs}, Xiao Ai Train Accuracy: {train_accuracy:.4f}'
        print(log_message)
    #print("max_accuracy:", max_accuracy)

#  Evaluation function.
def eval_model(model, data_loader, device):
    model.eval()
    corrects = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in data_loader:
            inputs, labels = inputs.to(device), labels.to(device).float().view(-1, 1)
            outputs = model(inputs)
            preds = outputs >= 0.5
            corrects += torch.sum(preds == labels).item()
            total += labels.size(0)
    accuracy = corrects / total
    return accuracy

class MyModel(nn.Module):
    def __init__(self):
        super(MyModel, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, 3)
        self.conv2 = nn.Conv2d(16, 16, 3)
        self.pool = nn.MaxPool2d(2, 1)
        self.fc1 = nn.Linear(10816, 128)
        self.fc2 = nn.Linear(128, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.reshape(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [5]:
model = MyModel()
summary(model, input_size=[1, 3, 32, 32])

Layer (type:depth-idx)                   Output Shape              Param #
MyModel                                  [1, 1]                    --
├─Conv2d: 1-1                            [1, 16, 30, 30]           448
├─ReLU: 1-2                              [1, 16, 30, 30]           --
├─MaxPool2d: 1-3                         [1, 16, 29, 29]           --
├─Conv2d: 1-4                            [1, 16, 27, 27]           2,320
├─ReLU: 1-5                              [1, 16, 27, 27]           --
├─MaxPool2d: 1-6                         [1, 16, 26, 26]           --
├─Linear: 1-7                            [1, 128]                  1,384,576
├─ReLU: 1-8                              [1, 128]                  --
├─Linear: 1-9                            [1, 1]                    129
Total params: 1,387,473
Trainable params: 1,387,473
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 3.48
Input size (MB): 0.01
Forward/backward pass size (MB): 0.21
Params size (MB): 5.55
Estimated Tota

### Eval

In [ ]:
#  Set up device.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f'Using device: {device}')

#  Create model, loss function, and optimizer.
model = MyModel().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4)
epochs = 10
#Train the model and log the process.
train_model(model, train_loader, test_loader, criterion, optimizer,device, num_epochs= epochs)

torch.save(model.state_dict(), 'lr-4.pth')

Using device: cuda
Epoch 1/10, Xiao Ai Train Accuracy: 0.7218
Epoch 2/10, Xiao Ai Train Accuracy: 0.7125
Epoch 3/10, Xiao Ai Train Accuracy: 0.7225
Epoch 4/10, Xiao Ai Train Accuracy: 0.7219
Epoch 5/10, Xiao Ai Train Accuracy: 0.7431
Epoch 6/10, Xiao Ai Train Accuracy: 0.7409
Epoch 7/10, Xiao Ai Train Accuracy: 0.7548
Epoch 8/10, Xiao Ai Train Accuracy: 0.7519
Epoch 9/10, Xiao Ai Train Accuracy: 0.7582
Epoch 10/10, Xiao Ai Train Accuracy: 0.7534


In [7]:
import zipfile
import os

# Define the files to be packaged and the compressed file name.
files_to_zip = ['submission_model.py', 'submission_dic.pth']
zip_filename = 'submission.zip'

# Create a zip file
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for file in files_to_zip:
        # 将Add files to the zip file.
        zipf.write(file, os.path.basename(file))

print(f'{zip_filename} is created successfully!')

FileNotFoundError: [WinError 2] The system cannot find the file specified: 'submission_model.py'